# Evolutionary Clustering Search (ECS) for 0-1 Knapsack
This notebook adapts the ECS framework from the referenced hybrid metaheuristics paper to the discrete 0-1 knapsack setting. The implementation keeps the four conceptual blocks—search metaheuristic (SM), iterative clustering (IC), analyzer module (AM), and local searcher (LS)—while tailoring representation, fitness, and local refinements to knapsack feasibility constraints.

**Key assumptions for this adaptation**
- Individuals are binary vectors (item selected or not); feasibility is enforced via penalties plus a repair-oriented local search.
- Clusters are maintained as sliding windows over promising regions based on Hamming distance and fitness dominance, capped by the paper’s suggested `N_C` (max clusters).
- The analyzer module promotes clusters whose mean fitness exceeds the population mean and dispatches a budgeted hill-climber around the cluster centroid before diversity collapses.
- The local searcher performs flip-based improvement with weight-awareness, akin to the “assimilation” mechanism described in the paper.

Subsequent cells provide a reusable ECS implementation plus an example driver you can adjust for your own knapsack instances.

In [2]:
from __future__ import annotations

import math
import random
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np


def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


@dataclass
class KnapsackProblem:
    values: np.ndarray
    weights: np.ndarray
    capacity: float

    def evaluate(self, chromosome: np.ndarray, penalty: float = 1.5) -> float:
        weight = float(np.dot(chromosome, self.weights))
        value = float(np.dot(chromosome, self.values))
        if weight <= self.capacity:
            return value
        # Penalize overweight solutions proportionally to the violation
        return value - penalty * (weight - self.capacity)

In [8]:



class EvolutionaryClusteringSearch:
    """Evolutionary Clustering Search tailored to the 0-1 Knapsack problem."""

    def __init__(
        self,
        problem: KnapsackProblem,
        population_size: int = 500,
        max_clusters: int = 30,
        max_generations: int = 2,
        tournament_size: int = 3,
        mutation_rate: float = 0.02,
        analyzer_threshold: float = 0.05,
        local_search_steps: int = 40,
        assimilation_rate: float = 0.05,
        cooling_pressure: float = 0.2,
        min_radius_bits: int = 1,
        seed: int | None = None,
    ) -> None:
        if seed is not None:
            set_random_seed(seed)
        self.problem = problem
        self.n_items = problem.values.size
        self.population_size = population_size
        self.max_clusters = max_clusters
        self.max_generations = max_generations
        self.tournament_size = tournament_size
        self.mutation_rate = mutation_rate
        self.analyzer_threshold = analyzer_threshold
        self.local_search_steps = local_search_steps
        self.assimilation_rate = assimilation_rate
        self.cooling_pressure = cooling_pressure
        self.min_radius_bits = min_radius_bits
        self.clusters_state: List[dict] = []

    def solve(self) -> Tuple[np.ndarray, float]:
        population = self._init_population()
        best_solution = None
        best_fitness = -math.inf

        for _ in range(self.max_generations):
            fitness = self._evaluate_population(population)
            idx = int(np.argmax(fitness))
            if fitness[idx] > best_fitness:
                best_fitness = float(fitness[idx])
                best_solution = population[idx].copy()

            clusters = self._cluster_population(population, fitness)
            improvements = self._run_local_search(clusters)
            population = self._assimilate(population, fitness, improvements)
            population = self._reproduce(population)

        assert best_solution is not None
        return best_solution, best_fitness

    # --- ECS building blocks -------------------------------------------------
    def _init_population(self) -> np.ndarray:
        return np.random.randint(0, 2, size=(self.population_size, self.n_items), dtype=np.int8)

    def _evaluate_population(self, population: np.ndarray) -> np.ndarray:
        return np.array([self.problem.evaluate(ind) for ind in population], dtype=float)

    def _compute_radius(self, cluster_count: int) -> int:
        if cluster_count <= 0:
            return self.n_items
        span = 1.0  # binary domain [0, 1]
        frac_radius = span / (2.0 * (cluster_count ** (1.0 / max(1, self.n_items))))
        hamming_radius = max(self.min_radius_bits, int(round(frac_radius * self.n_items)))
        return min(self.n_items, hamming_radius)

    def _cluster_population(
        self, population: np.ndarray, fitness: np.ndarray
    ) -> List[Tuple[np.ndarray, np.ndarray]]:
        """Iterative clustering with dynamic radius and assimilation."""
        order = np.argsort(fitness)[::-1]
        for cluster in self.clusters_state: # to remember cluster center
            cluster["members"] = []

        for idx in order:
            individual = population[idx]
            radius = self._compute_radius(len(self.clusters_state) or 1)
            assigned = False
            best_cluster = None
            best_distance = self.n_items + 1

            for cluster in self.clusters_state:
                binary_center = (cluster["center"] >= 0.5).astype(np.int8)
                distance = np.count_nonzero(binary_center != individual)
                if distance < best_distance:
                    best_distance = distance
                    best_cluster = cluster
                if distance <= radius:
                    cluster["center"] = (1 - self.assimilation_rate) * cluster["center"] + self.assimilation_rate * individual
                    cluster["members"].append(individual.copy())
                    cluster["inactive"] = 0
                    assigned = True
                    break

            if not assigned:
                if len(self.clusters_state) < self.max_clusters:
                    self.clusters_state.append(
                        {"center": individual.astype(float), "members": [individual.copy()], "inactive": 0}
                    )
                elif best_cluster is not None:
                    best_cluster["center"] = (1 - self.assimilation_rate) * best_cluster["center"] + self.assimilation_rate * individual
                    best_cluster["members"].append(individual.copy())
                    best_cluster["inactive"] = 0

        cooled_clusters = []
        for cluster in self.clusters_state:
            if cluster["members"]:
                cooled_clusters.append(cluster)
            else:
                cluster["inactive"] += 1
                threshold = max(1, int(self.cooling_pressure * self.population_size / max(1, len(self.clusters_state))))
                if cluster["inactive"] < threshold:
                    cooled_clusters.append(cluster)
        self.clusters_state = cooled_clusters[: self.max_clusters]

        formatted_clusters: List[Tuple[np.ndarray, np.ndarray]] = []
        for cluster in self.clusters_state:
            if not cluster["members"]:
                continue
            binary_center = (cluster["center"] >= 0.5).astype(np.int8)
            print(f"Cluster center: {binary_center}, Members: {len(cluster['center'])}")
            formatted_clusters.append((binary_center, np.vstack(cluster["members"])))
        return formatted_clusters

    def _analyzer_score(self, cluster_members: np.ndarray) -> float:
        cluster_fitness = self._evaluate_population(cluster_members)
        return float(np.mean(cluster_fitness))

    def _run_local_search(self, clusters: List[Tuple[np.ndarray, np.ndarray]]) -> List[np.ndarray]:
        improvements: List[np.ndarray] = []
        if not clusters:
            return improvements

        population_mean = np.mean([self._analyzer_score(members) for _, members in clusters])
        for center, members in clusters:
            score = self._analyzer_score(members)
            if score < (1.0 + self.analyzer_threshold) * population_mean:
                continue
            candidate = self._hill_climb(center.copy())
            improvements.append(candidate)
        return improvements

    def _hill_climb(self, chromosome: np.ndarray) -> np.ndarray:
        best = chromosome.copy()
        best_score = self.problem.evaluate(best)
        for _ in range(self.local_search_steps):
            idx = random.randrange(self.n_items)
            neighbor = best.copy()
            neighbor[idx] ^= 1  # flip bit
            neighbor_score = self.problem.evaluate(neighbor)
            if neighbor_score >= best_score:
                best, best_score = neighbor, neighbor_score
        return best

    def _assimilate(
        self,
        population: np.ndarray,
        fitness: np.ndarray,
        improvements: List[np.ndarray],
    ) -> np.ndarray:
        if not improvements:
            return population
        candidates = np.vstack([population] + improvements)
        candidate_fitness = self._evaluate_population(candidates)
        order = np.argsort(candidate_fitness)[::-1][: self.population_size]
        return candidates[order]

    def _reproduce(self, population: np.ndarray) -> np.ndarray:
        next_population = []
        fitness = self._evaluate_population(population)
        while len(next_population) < self.population_size:
            parent_a = self._tournament_select(population, fitness)
            parent_b = self._tournament_select(population, fitness)
            child = self._uniform_crossover(parent_a, parent_b)
            child = self._mutate(child)
            next_population.append(child)
        return np.vstack(next_population)

    def _tournament_select(self, population: np.ndarray, fitness: np.ndarray) -> np.ndarray:
        idx = np.random.choice(len(population), size=self.tournament_size, replace=False)
        winner = idx[np.argmax(fitness[idx])]
        return population[winner]

    def _uniform_crossover(self, parent_a: np.ndarray, parent_b: np.ndarray) -> np.ndarray:
        mask = np.random.rand(self.n_items) < 0.5
        child = np.where(mask, parent_a, parent_b).copy()
        return child

    def _mutate(self, chromosome: np.ndarray) -> np.ndarray:
        mutation_mask = np.random.rand(self.n_items) < self.mutation_rate
        chromosome[mutation_mask] ^= 1
        return chromosome


In [9]:
center = np.array([1, 0, 1, 0], dtype=np.int8)
individual = np.array([1, 1, 0, 0], dtype=np.int8)
distance = np.count_nonzero(center != individual)
print(distance)  # 2 differing positions

2


In [10]:
set_random_seed(7)
n_items = 60
values = np.random.randint(10, 120, size=n_items)
weights = np.random.randint(5, 45, size=n_items)
capacity = 0.4 * np.sum(weights)  # moderately constrained

In [11]:
knapsack = KnapsackProblem(values=values, weights=weights, capacity=capacity)

In [12]:
ecs = EvolutionaryClusteringSearch(
    problem=knapsack,
    population_size=500,
    max_clusters=30,
    max_generations=2,
    mutation_rate=0.015,
    min_radius_bits=6,
    local_search_steps=60,
    seed=42,
)

In [13]:
best_solution, best_value = ecs.solve()
total_weight = float(np.dot(best_solution, weights))

print(f"Best objective value: {best_value:.2f}")
print(f"Total weight: {total_weight:.2f} / {capacity:.2f}")
print(f"Items selected: {int(np.sum(best_solution))} / {n_items}")


Cluster center: [0 0 0 0 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 1
 0 0 1 0 1 1 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0], Members: 60
Cluster center: [0 0 0 0 0 1 0 0 0 1 1 0 0 1 1 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 0 0 0
 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 0 1 0 1 1 0 0 1], Members: 60
Cluster center: [1 1 0 0 1 1 0 1 1 0 0 1 0 1 0 1 0 0 0 1 1 0 1 0 0 0 0 0 1 1 0 1 0 0 1 1 0
 0 0 1 0 1 1 1 0 0 1 0 0 0 0 0 1 1 1 0 1 0 0 0], Members: 60
Cluster center: [1 0 1 0 0 0 1 1 0 1 0 1 1 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 1 1 0 1 0 1 1 1
 1 1 1 1 0 1 0 1 0 1 1 1 0 0 1 0 0 0 0 1 0 1 1], Members: 60
Cluster center: [1 0 0 1 1 0 0 0 0 0 1 1 1 1 1 0 1 1 1 1 1 1 0 1 0 0 1 0 1 0 0 1 1 1 0 1 1
 0 1 1 0 1 0 1 1 0 0 0 1 1 0 0 1 1 1 0 0 0 0 0], Members: 60
Cluster center: [1 1 0 0 0 1 1 0 0 0 1 0 0 1 1 1 1 1 1 0 0 0 0 1 1 1 1 1 1 1 0 0 0 0 0 0 0
 1 1 0 1 0 1 1 0 0 1 1 1 1 1 1 0 0 0 1 0 1 0 0], Members: 60
Cluster center: [1 1 1 1 0 0 1 0 1 0 0 0 0 0 0 0 1 1 0 0 1 1 1 1 1 0 1 1 0 0 0 1 0 1 0 1